In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

## Example code given: a Python translation of the MATLAB example. It:
   - Loads the three channels for the selected **slice (1–8)** and **data type (good or bad)**.
   - Converts k-space into images using the inverse Fourier transform.
   - Combines the images using root sum of squares and applies brightness compensation.
   - Crops the eye to **128 × 128 pixels** and adjusts contrast.
   - Plots the combined eye image and channel 1’s k-space.

In [ ]:
# Example code given

# Script to translate the K-spcae images into spatial eye images

slice_number = 1     # 1 to 8
data_type = "good"   # "good" or "bad"

folder = f"MRI_datasets/Slice{slice_number}/{data_type.capitalize()}Data/"
kspace1 = loadmat(folder + f"slice{slice_number}_channel1.mat")[f"slice{slice_number}_channel1_{data_type}Data"]
kspace2 = loadmat(folder + f"slice{slice_number}_channel2.mat")[f"slice{slice_number}_channel2_{data_type}Data"]
kspace3 = loadmat(folder + f"slice{slice_number}_channel3.mat")[f"slice{slice_number}_channel3_{data_type}Data"]

# 1. X - dimension of the K-Space data    - 128
# 2. Y - dimension of the K-Space data    - 512

# IFFT of k-space data
# channel 1
image1 = np.fft.ifftshift(np.fft.ifft2(kspace1), axes=0)
# channel 2
image2 = np.fft.ifftshift(np.fft.ifft2(kspace2), axes=0)
# channel 3
image3 = np.fft.ifftshift(np.fft.ifft2(kspace3), axes=0)

# clear compensation, preparation, based on fourier transformed blinked
# k-space data (Data_raw)
clear_comp = np.linspace(10, 0.1, image1.shape[1]) ** 2
clear_matrix = np.tile(clear_comp, (image1.shape[0], 1))

# combine 3 channels sum of squares and add clear compensation
eye_raw = np.sqrt(np.abs(image1) ** 2 + np.abs(image2) ** 2 + np.abs(image3) ** 2)
eye_raw = eye_raw * clear_matrix

# crop images because we are only interested in eye. Make it square
# 128 x 128

# Visualize the images.

# image
eye_visualize = eye_raw[187:315, :].copy()  # crop coordinates

# For better visualization and contrast of the eye images, histogram based
# compensation will be done
std_within = 0.995
# set maximum intensity to contain 99.5 % of intensity values per image
bin_centers = np.linspace(0, eye_visualize.max(), 1000)
middle_edges = (bin_centers[:-1] + bin_centers[1:]) / 2
bin_edges = np.concatenate(([-np.inf], middle_edges, [np.inf]))
counts, _ = np.histogram(eye_visualize, bins=bin_edges)
pixel_fraction = np.cumsum(counts) / counts.sum()
first_bin = np.where(pixel_fraction > std_within)[0][0]
thresh = bin_centers[first_bin]

# set threshold value to 65536
eye_visualize = eye_visualize * 65536 / thresh
eye_visualize = np.floor(eye_visualize + 0.5)
eye_visualize = np.clip(eye_visualize, 0, 65535).astype(np.uint16)

# plotting scripts
plt.figure()
plt.imshow(eye_visualize, cmap="gray", interpolation="nearest")
plt.axis("image")
plt.axis("off")
plt.show()

# Spatial frequency observations
plt.figure()
plt.imshow(100 * np.log(np.maximum(np.abs(kspace1), 1e-10)), aspect="auto",
           interpolation="nearest")
plt.xlabel("Horizontal frequency bins")
plt.ylabel("Vertical frequency bins")
plt.show()

## Good Images (No Outliers)
Plot three sub-images as an illustration that fusion is needed